# Notebook 04: Job Description Corpus Preparation

## Objective
Inspect the LinkedIn job postings dataset, understand the schema and table relationships,
assess data quality, and build a clean representative job description corpus across six
target domains for use in ATS scoring, semantic matching, and skill gap analysis.

## Methodology
1. Inspect raw schema and row counts for all five dataset files
2. Understand join relationships between postings, skills, and industries
3. Assess data quality in the main postings table
4. Design and apply domain assignment logic
5. Filter to a representative corpus of job descriptions
6. Apply skill normalization using the existing ESCO pipeline
7. Export three artifacts for downstream notebooks

## Assumptions
- postings.csv is the primary table; all other files join to it via job_id
- skills.csv in mappings is a lookup for skill_abr codes used in job_skills.csv
- industries.csv in mappings is a lookup for industry codes used in job_industries.csv
- Domain assignment will be rule-based using job title keywords
- The target corpus is 200 to 300 postings covering all six domains

## Inputs
- ../Dataset/postings.csv
- ../Dataset/job_skills.csv
- ../Dataset/job_industries.csv
- ../Dataset/industries.csv
- ../Dataset/skills.csv

## Outputs
- ../outputs/curated_job_descriptions.csv
- ../outputs/job_title_taxonomy.csv
- ../outputs/domain_distribution_summary.csv

In [ ]:
# importing libraries
import pandas as pd
import os

# loading all five dataset files
# postings.csv is large — loading with low_memory=False to avoid mixed dtype warnings
postings = pd.read_csv("../Dataset/postings.csv", low_memory=False)
job_skills = pd.read_csv("../Dataset/job_skills.csv")
job_industries = pd.read_csv("../Dataset/job_industries.csv")
industries_lookup = pd.read_csv("../Dataset/industries.csv")
skills_lookup = pd.read_csv("../Dataset/skills.csv")

# inspecting shape of each file
print("=== ROW AND COLUMN COUNTS ===")
print(f"postings:          {postings.shape}")
print(f"job_skills:        {job_skills.shape}")
print(f"job_industries:    {job_industries.shape}")
print(f"industries_lookup: {industries_lookup.shape}")
print(f"skills_lookup:     {skills_lookup.shape}")
print()

# inspecting column names for each file
print(" POSTINGS COLUMNS ")
for col in postings.columns:
    print(f"  {col}: {postings[col].dtype}")
print()

print(" JOB_SKILLS COLUMNS ")
for col in job_skills.columns:
    print(f"  {col}: {job_skills[col].dtype}")
print()

print(" JOB_INDUSTRIES COLUMNS ")
for col in job_industries.columns:
    print(f"  {col}: {job_industries[col].dtype}")
print()

print(" INDUSTRIES_LOOKUP COLUMNS ")
for col in industries_lookup.columns:
    print(f"  {col}: {industries_lookup[col].dtype}")
print()

print("=== SKILLS_LOOKUP COLUMNS ===")
for col in skills_lookup.columns:
    print(f"  {col}: {skills_lookup[col].dtype}")

=== ROW AND COLUMN COUNTS ===
postings:          (123849, 31)
job_skills:        (213768, 2)
job_industries:    (164808, 2)
industries_lookup: (422, 2)
skills_lookup:     (35, 2)

=== POSTINGS COLUMNS ===
  job_id: int64
  company_name: object
  title: object
  description: object
  max_salary: float64
  pay_period: object
  location: object
  company_id: float64
  views: float64
  med_salary: float64
  min_salary: float64
  formatted_work_type: object
  applies: float64
  original_listed_time: float64
  remote_allowed: float64
  job_posting_url: object
  application_url: object
  application_type: object
  expiry: float64
  closed_time: float64
  formatted_experience_level: object
  skills_desc: object
  listed_time: float64
  posting_domain: object
  sponsored: int64
  work_type: object
  currency: object
  compensation_type: object
  normalized_salary: float64
  zip_code: float64
  fips: float64

=== JOB_SKILLS COLUMNS ===
  job_id: int64
  skill_abr: object

=== JOB_INDUSTRIES COLU

In [3]:
# inspecting key columns in postings for data quality
print("=== POSTINGS - MISSING VALUE COUNTS (selected columns) ===")
key_cols = [
    "job_id", "title", "description", "skills_desc",
    "formatted_experience_level", "formatted_work_type",
    "location", "company_name"
]
for col in key_cols:
    null_count = postings[col].isnull().sum()
    null_pct = round(null_count / len(postings) * 100, 2)
    print(f"  {col:<35} nulls: {null_count:>7}  ({null_pct}%)")
print()

# inspecting experience level vocabulary
print(" FORMATTED_EXPERIENCE_LEVEL VALUES ")
print(postings["formatted_experience_level"].value_counts(dropna=False).to_string())
print()

# inspecting work type vocabulary
print(" FORMATTED_WORK_TYPE VALUES ")
print(postings["formatted_work_type"].value_counts(dropna=False).to_string())
print()

# sample job titles to understand naming patterns
print(" SAMPLE JOB TITLES (first 30 unique) ")
sample_titles = postings["title"].dropna().unique()[:30]
for t in sample_titles:
    print(f"  {t}")
print()

# inspecting skills_lookup to understand the 35 skill tags
print(" SKILLS LOOKUP - ALL 35 TAGS ")
print(skills_lookup.to_string(index=False))
print()

# checking how many postings have at least one skill tag
postings_with_skills = job_skills["job_id"].nunique()
print(f"Postings with at least one skill tag: {postings_with_skills}")
print(f"Total postings:                       {len(postings)}")
print(f"Coverage:                             {round(postings_with_skills / len(postings) * 100, 2)}%")

=== POSTINGS - MISSING VALUE COUNTS (selected columns) ===
  job_id                              nulls:       0  (0.0%)
  title                               nulls:       0  (0.0%)
  description                         nulls:       7  (0.01%)
  skills_desc                         nulls:  121410  (98.03%)
  formatted_experience_level          nulls:   29409  (23.75%)
  formatted_work_type                 nulls:       0  (0.0%)
  location                            nulls:       0  (0.0%)
  company_name                        nulls:    1719  (1.39%)

 FORMATTED_EXPERIENCE_LEVEL VALUES 
formatted_experience_level
Mid-Senior level    41489
Entry level         36708
NaN                 29409
Associate            9826
Director             3746
Internship           1449
Executive            1222

 FORMATTED_WORK_TYPE VALUES 
formatted_work_type
Full-time     98814
Contract      12117
Part-time      9696
Temporary      1190
Internship      983
Volunteer       562
Other           487

 SAMPLE JO

## Section 1: Schema and Data Quality Findings

### Key Findings
- description column is reliable (7 nulls across 123,849 postings)
- skills_desc is 98% null and will not be used
- formatted_experience_level has 23.75% nulls — not suitable as a required filter
- job_skills maps multiple skill tags per posting — coverage is effectively complete
- The 35 skill tags are coarse categorical labels, not granular skill tokens
- The dataset is broad and includes many irrelevant domains (healthcare, trades, food service)
  that must be filtered before corpus construction

### Join Strategy
- postings -> job_skills via job_id (one-to-many)
- postings -> job_industries via job_id (one-to-many)
- job_skills -> skills_lookup via skill_abr
- job_industries -> industries_lookup via industry_id

### Domain Assignment Strategy
Domain assignment will use a two-signal approach:
1. Skill tags from job_skills joined to skills_lookup (coarse signal)
2. Job title keyword matching (fine-grained signal, takes priority)

Title-based classification takes priority because skill tags are too coarse
to distinguish Data Science from IT, or Legal from Consulting.

In [4]:
# joining skill tags to postings for inspection
# resolving skill_abr to readable skill_name
job_skills_named = job_skills.merge(skills_lookup, on="skill_abr", how="left")

# aggregating skill tags per posting as a list
skills_per_posting = (
    job_skills_named.groupby("job_id")["skill_name"]
    .apply(list)
    .reset_index()
    .rename(columns={"skill_name": "skill_tags"})
)

print(f"Unique job_ids in job_skills: {job_skills['job_id'].nunique()}")
print(f"Unique job_ids in postings:   {postings['job_id'].nunique()}")
print()

# checking overlap
skills_ids = set(job_skills["job_id"].unique())
postings_ids = set(postings["job_id"].unique())

in_skills_not_postings = len(skills_ids - postings_ids)
in_postings_not_skills = len(postings_ids - skills_ids)

print(f"job_ids in job_skills but not in postings: {in_skills_not_postings}")
print(f"job_ids in postings but not in job_skills: {in_postings_not_skills}")
print()

# checking overall skill tag frequency across all postings
print("=== SKILL TAG FREQUENCY ACROSS ALL POSTINGS ===")
tag_freq = job_skills_named["skill_name"].value_counts()
print(tag_freq.to_string())
print()

# checking how many tags per posting on average
tags_per_posting = job_skills_named.groupby("job_id")["skill_name"].count()
print(f"Skill tags per posting - min/mean/max: "
      f"{tags_per_posting.min()} / "
      f"{round(tags_per_posting.mean(), 2)} / "
      f"{tags_per_posting.max()}")

Unique job_ids in job_skills: 126807
Unique job_ids in postings:   123849

job_ids in job_skills but not in postings: 4711
job_ids in postings but not in job_skills: 1753

=== SKILL TAG FREQUENCY ACROSS ALL POSTINGS ===
skill_name
Information Technology    26137
Sales                     22475
Management                20861
Manufacturing             18185
Health Care Provider      17369
Business Development      14290
Engineering               13009
Other                     12608
Finance                    8540
Marketing                  5525
Accounting/Auditing        5461
Administrative             4860
Customer Service           4292
Project Management         3997
Analyst                    3858
Research                   2986
Human Resources            2647
Legal                      2371
Consulting                 2338
Education                  2290
Design                     2244
Training                   2243
General Business           1984
Quality Assurance          1791
A

In [5]:
# joining industry names to postings
job_industries_named = job_industries.merge(industries_lookup, on="industry_id", how="left")

# checking overlap with postings
industry_ids = set(job_industries["job_id"].unique())
print(f"Unique job_ids in job_industries: {job_industries['job_id'].nunique()}")
print(f"job_ids in job_industries but not in postings: {len(industry_ids - postings_ids)}")
print(f"job_ids in postings but not in job_industries: {len(postings_ids - industry_ids)}")
print()

# aggregating industry names per posting
industries_per_posting = (
    job_industries_named.groupby("job_id")["industry_name"]
    .apply(list)
    .reset_index()
    .rename(columns={"industry_name": "industry_tags"})
)

# top 40 industries by posting count
print("=== TOP 40 INDUSTRIES BY POSTING COUNT ===")
industry_freq = job_industries_named["industry_name"].value_counts().head(40)
print(industry_freq.to_string())
print()

# industries per posting distribution
industries_per_count = job_industries_named.groupby("job_id")["industry_name"].count()
print(f"Industries per posting - min/mean/max: "
      f"{industries_per_count.min()} / "
      f"{round(industries_per_count.mean(), 2)} / "
      f"{industries_per_count.max()}")

Unique job_ids in job_industries: 127125
job_ids in job_industries but not in postings: 4712
job_ids in postings but not in job_industries: 1436

=== TOP 40 INDUSTRIES BY POSTING COUNT ===
industry_name
Hospitals and Health Care                                18326
Retail                                                   11033
IT Services and IT Consulting                            10396
Staffing and Recruiting                                   9005
Financial Services                                        8535
Software Development                                      5091
Manufacturing                                             3689
Construction                                              3445
Banking                                                   2923
Insurance                                                 2673
Pharmaceutical Manufacturing                              2469
Hospitality                                               2455
Telecommunications                       

## Section 2: Domain Assignment Strategy

### Signal Availability Summary
Three signals are available for domain assignment:
- Job title keywords (available for all 123,849 postings)
- Skill tags from job_skills (available for 122,096 postings, 35 coarse categories)
- Industry tags from job_industries (available for 122,413 postings, 422 categories)

### Domain Assignment Logic
Title-based keyword matching is the primary signal and takes priority in all cases.
Skill tags and industry tags are used as confirmation signals to reduce false positives.

The six target domains and their classification rules are defined below.

**Information Technology**
Title keywords: software engineer, developer, devops, cloud engineer, sysadmin,
  network engineer, backend, frontend, full stack, site reliability, platform engineer,
  infrastructure, systems engineer, IT support, cybersecurity, security engineer
Skill tag confirmation: Information Technology
Industry tag confirmation: IT Services and IT Consulting, Software Development,
  Technology Information and Internet, Information Services, Telecommunications

**Data Science**
Title keywords: data scientist, data analyst, data engineer, machine learning engineer,
  ML engineer, AI engineer, NLP engineer, analytics engineer, business intelligence,
  BI developer, research scientist
Skill tag confirmation: Analyst, Research
Industry tag confirmation: IT Services and IT Consulting, Software Development,
  Technology Information and Internet, Biotechnology Research

**Human Resources**
Title keywords: HR, human resources, talent acquisition, recruiter, recruiting,
  people operations, HRBP, HR business partner, HR manager, HR generalist,
  compensation, benefits manager, workforce planning
Skill tag confirmation: Human Resources
Industry tag confirmation: Staffing and Recruiting, Human Resources (if present)

**Legal**
Title keywords: attorney, lawyer, counsel, legal, paralegal, compliance officer,
  contract, litigation, associate attorney, legal counsel, general counsel
Skill tag confirmation: Legal
Industry tag confirmation: Law Practice

**Engineering**
Title keywords: mechanical engineer, civil engineer, structural engineer,
  electrical engineer, process engineer, manufacturing engineer, quality engineer,
  chemical engineer, aerospace engineer, industrial engineer, embedded engineer,
  hardware engineer
Skill tag confirmation: Engineering, Manufacturing, Quality Assurance
Industry tag confirmation: Civil Engineering, Industrial Machinery Manufacturing,
  Defense and Space Manufacturing, Chemical Manufacturing,
  Appliances Electrical and Electronics Manufacturing

**Management**
Title keywords: program manager, project manager, operations manager, general manager,
  director of operations, VP of, head of, chief of staff, strategy manager,
  engagement manager, delivery manager, account manager, business analyst
Skill tag confirmation: Management, Project Management, Strategy/Planning
Industry tag confirmation: Business Consulting and Services

### Priority Rules
1. If a title matches Data Science keywords, assign Data Science regardless of skill tag
2. If a title matches Legal keywords, assign Legal regardless of industry
3. If a title matches HR keywords, assign HR regardless of industry
4. Engineering title keywords take priority over Manufacturing industry tag
   (to avoid assigning production line roles to Engineering domain)
5. If no title keyword matches, fall back to skill tag only for IT and Management
6. Postings with no matching signal are excluded

### Quality Filters Applied Before Domain Assignment
- Drop postings with null description
- Keep Full-time and Contract work types only
- Require description length of at least 200 characters
- Exclude internship titles (contain "intern" as standalone word)

In [6]:
# building the master joined table
# joining skill tags and industry tags to postings
# left join preserves all postings; unmatched skill/industry records are dropped naturally

postings_with_skills = postings.merge(skills_per_posting, on="job_id", how="left")
postings_joined = postings_with_skills.merge(industries_per_posting, on="job_id", how="left")

print(f"Postings after joins: {len(postings_joined)}")
print(f"Columns: {postings_joined.shape[1]}")
print()

# applying baseline quality filters
# keeping only full-time and contract roles
work_type_mask = postings_joined["formatted_work_type"].isin(["Full-time", "Contract"])

# dropping null descriptions
desc_mask = postings_joined["description"].notna()

# requiring minimum description length
desc_length_mask = postings_joined["description"].str.len() >= 200

# excluding internship titles
intern_mask = ~postings_joined["title"].str.lower().str.contains(
    r'\bintern\b', regex=True, na=False
)

filtered = postings_joined[work_type_mask & desc_mask & desc_length_mask & intern_mask].copy()
filtered = filtered.reset_index(drop=True)

print(f"After work type filter (Full-time + Contract): {work_type_mask.sum()}")
print(f"After null description filter:                 {desc_mask.sum()}")
print(f"After description length filter (>=200 chars): {desc_length_mask.sum()}")
print(f"After internship exclusion:                    {intern_mask.sum()}")
print(f"After all baseline filters combined:           {len(filtered)}")
print()

# checking description length distribution in filtered set
desc_lengths = filtered["description"].str.len()
print(f"Description length - min/mean/max: "
      f"{desc_lengths.min()} / "
      f"{round(desc_lengths.mean())} / "
      f"{desc_lengths.max()}")

Postings after joins: 123849
Columns: 33

After work type filter (Full-time + Contract): 110931
After null description filter:                 123842
After description length filter (>=200 chars): 123222
After internship exclusion:                    122571
After all baseline filters combined:           109874

Description length - min/mean/max: 200 / 3803 / 23201


In [8]:
# domain assignment using title keyword matching as primary signal
# skill tags and industry tags as secondary confirmation

def assign_domain(row):
    title = str(row["title"]).lower()
    skill_tags = [str(t).lower() for t in row["skill_tags"]] if isinstance(row["skill_tags"], list) else []
    industry_tags = [str(t).lower() for t in row["industry_tags"]] if isinstance(row["industry_tags"], list) else []

    # data science — must come before IT to avoid misclassification
    ds_title_keywords = [
        "data scientist", "data analyst", "data engineer", "machine learning",
        "ml engineer", "ai engineer", "nlp engineer", "analytics engineer",
        "business intelligence", "bi developer", "bi analyst", "research scientist",
        "deep learning", "computer vision"
    ]
    if any(kw in title for kw in ds_title_keywords):
        return "Data Science"

    # legal
    legal_title_keywords = [
        "attorney", "lawyer", "counsel", "paralegal", "litigation",
        "legal analyst", "legal associate", "legal counsel", "general counsel",
        "compliance officer", "compliance analyst", "contract manager"
    ]
    if any(kw in title for kw in legal_title_keywords):
        return "Legal"

    # human resources
    hr_title_keywords = [
        "human resources", " hr ", "hr manager", "hr business partner",
        "hr generalist", "hr director", "hr analyst", "hrbp",
        "talent acquisition", "recruiter", "recruiting", "people operations",
        "people partner", "compensation", "benefits manager", "workforce planning",
        "talent management"
    ]
    if any(kw in title for kw in hr_title_keywords):
        return "HR"

    # engineering — specific engineering disciplines to avoid catching generic titles
    eng_title_keywords = [
        "mechanical engineer", "civil engineer", "structural engineer",
        "electrical engineer", "process engineer", "manufacturing engineer",
        "quality engineer", "chemical engineer", "aerospace engineer",
        "industrial engineer", "embedded engineer", "hardware engineer",
        "controls engineer", "systems engineer", "field engineer",
        "reliability engineer", "materials engineer", "test engineer"
    ]
    if any(kw in title for kw in eng_title_keywords):
        return "Engineering"

    # information technology
    it_title_keywords = [
        "software engineer", "software developer", "devops", "cloud engineer",
        "backend engineer", "frontend engineer", "full stack", "fullstack",
        "site reliability", "platform engineer", "infrastructure engineer",
        "network engineer", "sysadmin", "systems administrator",
        "cybersecurity", "security engineer", "it support", "it analyst",
        "it manager", "it director", "solution architect", "solutions architect",
        "application developer", "mobile developer", "ios developer",
        "android developer", "web developer", "database administrator",
        "dba", "erp", "salesforce", "sap consultant"
    ]
    if any(kw in title for kw in it_title_keywords):
        return "IT"

    # management
    mgmt_title_keywords = [
        "program manager", "project manager", "operations manager",
        "general manager", "director of operations", "vp of", "head of",
        "chief of staff", "strategy manager", "engagement manager",
        "delivery manager", "account manager", "product manager",
        "business analyst", "management consultant", "scrum master",
        "agile coach", "pmo"
    ]
    if any(kw in title for kw in mgmt_title_keywords):
        return "Management"

    # skill tag fallback for IT and Management only
    if "information technology" in skill_tags:
        return "IT"
    if "management" in skill_tags and "human resources" not in skill_tags:
        return "Management"
    if "legal" in skill_tags:
        return "Legal"
    if "human resources" in skill_tags:
        return "HR"
    if "engineering" in skill_tags or "quality assurance" in skill_tags:
        return "Engineering"

    return None

# applying domain assignment
filtered["domain"] = filtered.apply(assign_domain, axis=1)

# checking domain distribution
print(" DOMAIN ASSIGNMENT RESULTS ")
domain_counts = filtered["domain"].value_counts(dropna=False)
print(domain_counts.to_string())
print()

total_assigned = filtered["domain"].notna().sum()
total_unassigned = filtered["domain"].isna().sum()
print(f"Total assigned:   {total_assigned}")
print(f"Total unassigned: {total_unassigned}")
print(f"Assignment rate:  {round(total_assigned / len(filtered) * 100, 2)}%")

 DOMAIN ASSIGNMENT RESULTS 
domain
None            54936
Management      21977
IT              20562
Engineering      5749
HR               2628
Legal            2610
Data Science     1412

Total assigned:   54938
Total unassigned: 54936
Assignment rate:  50.0%


In [9]:
# inspecting management titles to identify noise
print("=== SAMPLE MANAGEMENT TITLES (100 random) ===")
mgmt_sample = (
    filtered[filtered["domain"] == "Management"]["title"]
    .sample(100, random_state=42)
    .sort_values()
)
for t in mgmt_sample:
    print(f"  {t}")
print()

# inspecting data science titles to check coverage
print("=== ALL UNIQUE DATA SCIENCE TITLES (sorted) ===")
ds_titles = (
    filtered[filtered["domain"] == "Data Science"]["title"]
    .str.strip()
    .unique()
)
ds_titles_sorted = sorted(ds_titles)
for t in ds_titles_sorted[:80]:
    print(f"  {t}")
print(f"  ... ({len(ds_titles_sorted)} total unique titles)")

=== SAMPLE MANAGEMENT TITLES (100 random) ===
  $10k Sign -On-EXECUTIVE CHEF ( PROVIDENCE ST JOHNS HOSP) SANTA MONICA CA
  3rd Shift Bilingual Production Supervisor
  AN-Assistant Manager-A00027
  ASRS Maintenance Technician II - 3rd shift - $2000 signing bonus
  ASSISTANT FABRICATION OPERATIONS MANAGER
  Analyst, Supply Chain
  Area Operations Coordinator
  Assembler/Tester 1 (BTP)
  Assembly Cell Operator
  Automotive Maintenance Technician
  Building Construction Project Manager / Estimator
  Business Analyst
  CDL Truck Driver
  Carpenter Journeyman
  Chemical Operator
  Collision Estimator
  Community Engagement Manager
  Community Manager
  Construction Manager
  Custodian (37.5 Hour) Office/On-site #240405-0429TC-002
  Customer Service Associates
  DOT 16+ Bus Driver (CDL Required) - Detroit Metro Airport - Full Time
  Diesel Mechanic/Technician I - Experienced
  Digital Product Manager Lead Sr.
  Director of Finance And Accounting
  Director of Partner Services
  Distribution C

In [10]:
# identifying which keywords are responsible for the management noise
# checking title match for each keyword independently

mgmt_keywords = [
    "program manager", "project manager", "operations manager",
    "general manager", "director of operations", "vp of", "head of",
    "chief of staff", "strategy manager", "engagement manager",
    "delivery manager", "account manager", "product manager",
    "business analyst", "management consultant", "scrum master",
    "agile coach", "pmo"
]

mgmt_filtered = filtered[filtered["domain"] == "Management"].copy()

print("=== MANAGEMENT KEYWORD HIT COUNTS ===")
for kw in mgmt_keywords:
    count = mgmt_filtered["title"].str.lower().str.contains(kw, regex=False).sum()
    print(f"  {kw:<30} {count:>6}")
print()

# checking how many management postings have IT or Data Science skill tags
# these are likely legitimate technical PM or BA roles
it_tag_in_mgmt = mgmt_filtered["skill_tags"].apply(
    lambda x: "Information Technology" in x if isinstance(x, list) else False
).sum()
print(f"Management postings with IT skill tag:    {it_tag_in_mgmt}")

# checking industry tags in management postings to see dominant industries
mgmt_industry_flat = (
    mgmt_filtered["industry_tags"]
    .dropna()
    .explode()
    if hasattr(mgmt_filtered["industry_tags"], "explode")
    else mgmt_filtered["industry_tags"].apply(
        lambda x: x if isinstance(x, list) else []
    ).explode()
)
print()
print("=== TOP 20 INDUSTRIES IN MANAGEMENT POSTINGS ===")
print(mgmt_industry_flat.value_counts().head(20).to_string())

=== MANAGEMENT KEYWORD HIT COUNTS ===
  program manager                   606
  project manager                  1933
  operations manager                470
  general manager                   371
  director of operations             58
  vp of                              67
  head of                           196
  chief of staff                     32
  strategy manager                   15
  engagement manager                 29
  delivery manager                   40
  account manager                  1001
  product manager                   502
  business analyst                  519
  management consultant              45
  scrum master                       74
  agile coach                         8
  pmo                                31

Management postings with IT skill tag:    2234

=== TOP 20 INDUSTRIES IN MANAGEMENT POSTINGS ===
industry_tags
Retail                                                 1790
Construction                                           1386
Staffing a

In [ ]:
# industries that indicate out-of-scope management roles
EXCLUDE_MGMT_INDUSTRIES = {
    "retail", "construction", "hospitality", "hospitals and health care",
    "restaurants", "food and beverage services", "food and beverage manufacturing",
    "motor vehicle manufacturing", "real estate", "wellness and fitness services",
    "transportation, logistics, supply chain and storage",
    "industrial machinery manufacturing", "retail apparel and fashion"
}

# tightened management keywords — senior and strategic roles only
MGMT_TITLE_KEYWORDS_TIGHT = [
    "program manager", "project manager", "operations manager",
    "general manager", "director of operations", "vp of", "head of",
    "chief of staff", "strategy manager", "engagement manager",
    "delivery manager", "product manager", "management consultant",
    "scrum master", "agile coach", "pmo", "portfolio manager",
    "transformation manager", "change manager"
]

# account manager and business analyst removed from title keywords
# they will only be captured via IT skill tag confirmation

def assign_domain_v2(row):
    title = str(row["title"]).lower()
    skill_tags = [str(t).lower() for t in row["skill_tags"]] if isinstance(row["skill_tags"], list) else []
    industry_tags = [str(t).lower() for t in row["industry_tags"]] if isinstance(row["industry_tags"], list) else []

    # data science first
    ds_keywords = [
        "data scientist", "data analyst", "data engineer", "machine learning",
        "ml engineer", "ai engineer", "nlp engineer", "analytics engineer",
        "business intelligence", "bi developer", "bi analyst",
        "research scientist", "deep learning", "computer vision"
    ]
    if any(kw in title for kw in ds_keywords):
        return "Data Science"

    # legal
    legal_keywords = [
        "attorney", "lawyer", "counsel", "paralegal", "litigation",
        "legal analyst", "legal associate", "legal counsel", "general counsel",
        "compliance officer", "compliance analyst", "contract manager"
    ]
    if any(kw in title for kw in legal_keywords):
        return "Legal"

    # human resources
    hr_keywords = [
        "human resources", " hr ", "hr manager", "hr business partner",
        "hr generalist", "hr director", "hr analyst", "hrbp",
        "talent acquisition", "recruiter", "recruiting manager",
        "people operations", "people partner", "compensation",
        "benefits manager", "workforce planning", "talent management"
    ]
    if any(kw in title for kw in hr_keywords):
        return "HR"

    # engineering
    eng_keywords = [
        "mechanical engineer", "civil engineer", "structural engineer",
        "electrical engineer", "process engineer", "manufacturing engineer",
        "quality engineer", "chemical engineer", "aerospace engineer",
        "industrial engineer", "embedded engineer", "hardware engineer",
        "controls engineer", "field engineer", "reliability engineer",
        "materials engineer", "test engineer"
    ]
    if any(kw in title for kw in eng_keywords):
        return "Engineering"

    # information technology
    it_keywords = [
        "software engineer", "software developer", "devops", "cloud engineer",
        "backend engineer", "frontend engineer", "full stack", "fullstack",
        "site reliability", "platform engineer", "infrastructure engineer",
        "network engineer", "sysadmin", "systems administrator",
        "cybersecurity", "security engineer", "it support", "it analyst",
        "it manager", "it director", "solution architect", "solutions architect",
        "application developer", "mobile developer", "ios developer",
        "android developer", "web developer", "database administrator",
        "erp consultant", "salesforce", "sap consultant"
    ]
    if any(kw in title for kw in it_keywords):
        return "IT"

    # management — title match plus industry exclusion check
    if any(kw in title for kw in MGMT_TITLE_KEYWORDS_TIGHT):
        if not any(ind in EXCLUDE_MGMT_INDUSTRIES for ind in industry_tags):
            return "Management"

    # skill tag fallbacks
    if "information technology" in skill_tags:
        return "IT"
    if "legal" in skill_tags:
        return "Legal"
    if "human resources" in skill_tags:
        return "HR"
    if "engineering" in skill_tags or "quality assurance" in skill_tags:
        return "Engineering"
    if "management" in skill_tags and "human resources" not in skill_tags:
        if not any(ind in EXCLUDE_MGMT_INDUSTRIES for ind in industry_tags):
            return "Management"

    return None

# applying revised assignment
filtered["domain"] = filtered.apply(assign_domain_v2, axis=1)

print(" REVISED DOMAIN ASSIGNMENT RESULTS ")
domain_counts = filtered["domain"].value_counts(dropna=False)
print(domain_counts.to_string())
print()
total_assigned = filtered["domain"].notna().sum()
print(f"Total assigned:   {total_assigned}")
print(f"Assignment rate:  {round(total_assigned / len(filtered) * 100, 2)}%")

=== REVISED DOMAIN ASSIGNMENT RESULTS ===
domain
None            64824
IT              21123
Management      11439
Engineering      5809
Legal            2640
HR               2627
Data Science     1412

Total assigned:   45050
Assignment rate:  41.0%


In [ ]:
# spot checking each domain pool before sampling
# printing 10 random titles per domain to assess assignment quality

import random
random.seed(42)

domains = ["IT", "Data Science", "HR", "Legal", "Engineering", "Management"]

for domain in domains:
    pool = filtered[filtered["domain"] == domain]["title"].dropna().tolist()
    sample_size = min(10, len(pool))
    sampled = random.sample(pool, sample_size)
    print(f" {domain.upper()} (pool size: {len(pool)})")
    for t in sorted(sampled):
        print(f"  {t}")
    print()

=== IT (pool size: 21123) ===
  Android Developer/iOS Developer
  Assistant Director of Applied Biostatistics (87824BR)
  Business Consultant / Sales Engineer (remote)
  ELECTRONICS ASSMEBLER - Chesterfield, MI
  Laboratory Sciences Professional, Open-Rank Entry and Intermediate
  Overnight Security Professional
  React/Java Full Stack Developer
  Senior Information Technology Auditor
  Senior Software Frontend Engineer, Moderation Review
  Technician (C) - Tires Plus

=== DATA SCIENCE (pool size: 1412) ===
  Analytics Engineer (L4/5) - Pricing and Forecasting
  Data Analyst
  Data Analyst - Power BI & SQL
  Data Analyst-Tx
  Data Engineer
  Data Scientist
  Data Scientist
  Health Data Scientist
  Junior Business Intelligence Developer
  Vice President of Data Engineering

=== HR (pool size: 2627) ===
  Associate Recruiter - Remote
  ExecutSr Benefits & Wellness Specialistive Project Specialist
  Football Performance Nutrition Fellowship
  HR Business Partner - Corporate Functions - U

In [13]:
def assign_domain_v3(row):
    title = str(row["title"]).lower()
    skill_tags = [str(t).lower() for t in row["skill_tags"]] if isinstance(row["skill_tags"], list) else []
    industry_tags = [str(t).lower() for t in row["industry_tags"]] if isinstance(row["industry_tags"], list) else []

    # data science — evaluated first to avoid IT absorption
    ds_keywords = [
        "data scientist", "data analyst", "data engineer", "machine learning",
        "ml engineer", "ai engineer", "nlp engineer", "analytics engineer",
        "business intelligence", "bi developer", "bi analyst",
        "research scientist", "deep learning", "computer vision"
    ]
    if any(kw in title for kw in ds_keywords):
        return "Data Science"

    # legal — using word boundary logic to avoid substring matches
    # counselor, counseling excluded deliberately
    legal_keywords = [
        "attorney", " lawyer", "paralegal", "litigation",
        "legal analyst", "legal associate", "legal counsel", "general counsel",
        "compliance officer", "compliance analyst", "contract manager",
        "legal assistant", "legal manager", "legal director"
    ]
    if any(kw in title for kw in legal_keywords):
        return "Legal"

    # human resources
    hr_keywords = [
        "human resources", " hr ", "hr manager", "hr business partner",
        "hr generalist", "hr director", "hr analyst", "hrbp",
        "talent acquisition", "recruiter", "recruiting manager",
        "people operations", "people partner", "compensation analyst",
        "benefits manager", "workforce planning", "talent management"
    ]
    if any(kw in title for kw in hr_keywords):
        return "HR"

    # engineering — specific disciplines only
    eng_keywords = [
        "mechanical engineer", "civil engineer", "structural engineer",
        "electrical engineer", "process engineer", "manufacturing engineer",
        "quality engineer", "chemical engineer", "aerospace engineer",
        "industrial engineer", "embedded engineer", "hardware engineer",
        "controls engineer", "field engineer", "reliability engineer",
        "materials engineer", "test engineer"
    ]
    if any(kw in title for kw in eng_keywords):
        return "Engineering"

    # information technology — title only, no skill tag fallback
    it_keywords = [
        "software engineer", "software developer", "devops", "cloud engineer",
        "backend engineer", "frontend engineer", "full stack", "fullstack",
        "site reliability", "platform engineer", "infrastructure engineer",
        "network engineer", "sysadmin", "systems administrator",
        "cybersecurity", "security engineer", "it support", "it analyst",
        "it manager", "it director", "solution architect", "solutions architect",
        "application developer", "mobile developer", "ios developer",
        "android developer", "web developer", "database administrator",
        "erp consultant", "salesforce developer", "sap consultant"
    ]
    if any(kw in title for kw in it_keywords):
        return "IT"

    # management — title match plus industry exclusion
    # skill tag fallback removed to eliminate production/operations noise
    mgmt_keywords = [
        "program manager", "project manager", "operations manager",
        "general manager", "director of operations", "vp of", "head of",
        "chief of staff", "strategy manager", "engagement manager",
        "delivery manager", "product manager", "management consultant",
        "scrum master", "agile coach", "pmo", "portfolio manager",
        "transformation manager", "change manager"
    ]
    if any(kw in title for kw in mgmt_keywords):
        if not any(ind in EXCLUDE_MGMT_INDUSTRIES for ind in industry_tags):
            return "Management"

    # hr and legal skill tag fallbacks only — these are tight enough to be safe
    if "legal" in skill_tags:
        return "Legal"
    if "human resources" in skill_tags:
        return "HR"

    return None

# applying final assignment
filtered["domain"] = filtered.apply(assign_domain_v3, axis=1)

print("FINAL DOMAIN ASSIGNMENT RESULTS")
print(filtered["domain"].value_counts(dropna=False).to_string())
print()
total_assigned = filtered["domain"].notna().sum()
print(f"Total assigned:   {total_assigned}")
print(f"Assignment rate:  {round(total_assigned / len(filtered) * 100, 2)}%")
print()

# spot checking legal and IT specifically for the noise fixes
print("LEGAL SAMPLE - 10 random titles")
legal_pool = filtered[filtered["domain"] == "Legal"]["title"].dropna().tolist()
for t in sorted(random.sample(legal_pool, min(10, len(legal_pool)))):
    print(f"  {t}")
print()

print("IT SAMPLE - 10 random titles")
it_pool = filtered[filtered["domain"] == "IT"]["title"].dropna().tolist()
for t in sorted(random.sample(it_pool, min(10, len(it_pool)))):
    print(f"  {t}")

FINAL DOMAIN ASSIGNMENT RESULTS
domain
None            94077
IT               3947
Management       3325
HR               2609
Legal            2287
Engineering      2217
Data Science     1412

Total assigned:   15797
Assignment rate:  14.38%

LEGAL SAMPLE - 10 random titles
  Attorney
  CARE Contract Manager
  Conflicts Attorney
  Family Law Attorney
  Firearms Technician
  IP Paralegal
  Legal Counsel, Commercial Transactions
  Litigation Attorney - Hybrid
  Personal Injury Attorney
  Senior Litigation Docketing Specialist

IT SAMPLE - 10 random titles
  DevOps Engineer (W2)
  Full Stack Engineer
  Network Engineer III
  REMOTE Senior Consultant (.NET Full Stack)
  SAP S4 Finance Consultant (SAP Finance – FICO. FICA Solution Architecture)
  Salesforce Developer
  Security Engineer (SPLUNK) | Remote US
  Senior Full Stack Engineer (NC)
  Senior Software Engineer, Full Stack (Java, AWS)
  Sr. Software Engineer, Application Administrator, Salesforce


## Section 3: Domain Pool Summary and Sampling Design

### Final Domain Pool Sizes
After three iterations of domain assignment refinement:

| Domain       | Pool Size |
|--------------|-----------|
| IT           | 3,947     |
| Management   | 3,325     |
| HR           | 2,609     |
| Legal        | 2,287     |
| Engineering  | 2,217     |
| Data Science | 1,412     |

All six domains have sufficient pool depth for the target corpus size.
Data Science is the smallest pool and sets the upper sampling bound per domain.

### Sampling Strategy
Target corpus: 250 postings total, approximately 40 to 42 per domain.
Sampling is stratified by domain. Within each domain, postings are selected
by descending description length to prefer substantive content over stub postings.
A minimum description length of 500 characters is applied at sampling time
as an additional quality gate above the 200-character baseline filter.

### Description Length as Quality Proxy
Mean description length across filtered postings is 3,803 characters.
Selecting by description length ensures the semantic matching pipeline
has adequate text to embed. Short descriptions compress embeddings and
reduce matching score variance, which was identified as a risk in memory_data_02.

In [14]:
# stratified sampling — 42 postings per domain
# selecting columns needed for downstream notebooks only
SAMPLE_PER_DOMAIN = 42

output_cols = [
    "job_id", "title", "company_name", "location",
    "description", "formatted_experience_level",
    "formatted_work_type", "domain", "skill_tags", "industry_tags"
]

# applying additional quality gate at sampling time
MIN_DESC_LENGTH = 500

domain_samples = []

for domain in ["IT", "Data Science", "HR", "Legal", "Engineering", "Management"]:
    pool = filtered[
        (filtered["domain"] == domain) &
        (filtered["description"].str.len() >= MIN_DESC_LENGTH)
    ].copy()

    # sorting by description length descending to prefer substantive postings
    pool = pool.sort_values("description", key=lambda x: x.str.len(), ascending=False)

    # sampling without replacement
    n = min(SAMPLE_PER_DOMAIN, len(pool))
    sample = pool.head(n * 3).sample(n=n, random_state=42)
    domain_samples.append(sample)
    print(f"{domain:<15} pool after length filter: {len(pool):>5}   sampled: {n}")

corpus = pd.concat(domain_samples, ignore_index=True)
corpus = corpus[output_cols].copy()

print()
print(f"Total corpus size: {len(corpus)}")
print()
print("Domain distribution in corpus:")
print(corpus["domain"].value_counts().to_string())
print()
print(f"Description length in corpus - min/mean/max: "
      f"{corpus['description'].str.len().min()} / "
      f"{round(corpus['description'].str.len().mean())} / "
      f"{corpus['description'].str.len().max()}")

IT              pool after length filter:  3836   sampled: 42
Data Science    pool after length filter:  1369   sampled: 42
HR              pool after length filter:  2587   sampled: 42
Legal           pool after length filter:  2239   sampled: 42
Engineering     pool after length filter:  2199   sampled: 42
Management      pool after length filter:  3289   sampled: 42

Total corpus size: 252

Domain distribution in corpus:
domain
IT              42
Data Science    42
HR              42
Legal           42
Engineering     42
Management      42

Description length in corpus - min/mean/max: 7105 / 9392 / 23201


## Section 4: Skill Extraction and ESCO Normalization

Skill tokens are extracted from two sources per posting:
1. skill_tags from job_skills — coarse categorical labels, used as domain signal only
2. skills_desc was 98% null and is excluded

Since the LinkedIn skill tags are too coarse for skill gap analysis,
skill tokens for matching are extracted directly from the description text
using the same keyword vocabulary established in Notebook 03.

The extraction strategy is vocabulary-driven: scan each description for
occurrences of the 35 canonical tokens from the ESCO normalization pipeline.
This ensures job description skill profiles use the same token representation
as candidate skill profiles, making gap analysis a direct set comparison.

This approach is intentionally conservative. Only tokens present in the
established canonical vocabulary are extracted. Novel tokens in job descriptions
are not added to the vocabulary at this stage — that decision is deferred
to the ATS scoring notebook where vocabulary expansion can be evaluated
in the context of scoring logic.

In [15]:
# loading ESCO normalization artifacts from notebook 03
esco_mapping = pd.read_csv("../outputs/esco_skill_mapping.csv")
correction_map_reload = pd.read_csv("../outputs/cleaned_skill_vocabulary.csv")

# building canonical token list from esco mapping
# these are the 35 canonical tokens that form the matching vocabulary
canonical_tokens = esco_mapping["canonical_token"].tolist()

# building normalization lookup — same logic as notebook 03
normalization_lookup = {}
for _, row in esco_mapping.iterrows():
    if row["token_category"] == "esco_mapped":
        normalization_lookup[row["canonical_token"]] = row["esco_preferred_label"]
    else:
        normalization_lookup[row["canonical_token"]] = row["canonical_token"]

print(f"Canonical tokens available for matching: {len(canonical_tokens)}")
print()

# extracting skill tokens from job description text
# scanning for each canonical token as a substring in lowercased description
def extract_skills_from_description(description):
    text = str(description).lower()
    matched = []
    for token in canonical_tokens:
        if token in text:
            matched.append(token)
    return matched

corpus["extracted_skills_raw"] = corpus["description"].apply(extract_skills_from_description)

# applying normalization lookup to get normalized skill profile
def normalize_skill_list(token_list):
    normalized = [normalization_lookup.get(t, t) for t in token_list]
    return sorted(set(normalized))

corpus["normalized_skills"] = corpus["extracted_skills_raw"].apply(normalize_skill_list)
corpus["skill_count"] = corpus["normalized_skills"].apply(len)

# storing as pipe-delimited strings for CSV export
corpus["extracted_skills_raw"] = corpus["extracted_skills_raw"].apply(lambda x: "|".join(x))
corpus["normalized_skills"] = corpus["normalized_skills"].apply(lambda x: "|".join(x))

# skill extraction summary
print("Skill extraction results:")
print(f"  Postings with at least one skill extracted: "
      f"{(corpus['skill_count'] > 0).sum()} / {len(corpus)}")
print(f"  Skill count per posting - min/mean/max: "
      f"{corpus['skill_count'].min()} / "
      f"{round(corpus['skill_count'].mean(), 2)} / "
      f"{corpus['skill_count'].max()}")
print()

# skill frequency across corpus
from collections import Counter
all_extracted = []
for skills in corpus["extracted_skills_raw"]:
    all_extracted.extend(skills.split("|") if skills else [])

skill_freq = Counter(all_extracted)
print("Top 20 skills extracted across corpus:")
for skill, count in skill_freq.most_common(20):
    print(f"  {skill:<35} {count}")
print()

# skill extraction by domain
print("Mean skills extracted per domain:")
print(corpus.groupby("domain")["skill_count"].mean().round(2).to_string())

Canonical tokens available for matching: 35

Skill extraction results:
  Postings with at least one skill extracted: 240 / 252
  Skill count per posting - min/mean/max: 0 / 3.59 / 15

Top 20 skills extracted across corpus:
  compliance                          127
  aws                                 115
  strategy                            105
  git                                 62
  cad                                 52
  recruitment                         50
  agile                               44
  sql                                 41
  python                              39
  machine learning                    35
  risk management                     33
  manufacturing                       33
  azure                               21
  java                                17
  linux                               13
  payroll                             13
  employee engagement                 13
  kubernetes                          12
  docker                            

In [16]:
# investigating zero-skill postings
zero_skill = corpus[corpus["skill_count"] == 0][["job_id", "title", "domain"]]
print("Postings with zero skills extracted:")
print(zero_skill.to_string(index=False))
print()

# investigating CAD frequency — should be Engineering-specific
cad_postings = corpus[corpus["extracted_skills_raw"].str.contains("cad", na=False)]
print(f"Postings with CAD extracted: {len(cad_postings)}")
print()
print("CAD by domain:")
print(cad_postings["domain"].value_counts().to_string())
print()

# checking a non-engineering example to confirm false positive
non_eng_cad = cad_postings[cad_postings["domain"] != "Engineering"].head(3)
for _, row in non_eng_cad.iterrows():
    desc_lower = str(row["description"]).lower()
    # finding context around 'cad'
    idx = desc_lower.find("cad")
    context = desc_lower[max(0, idx-40):idx+40]
    print(f"Title: {row['title']} | Domain: {row['domain']}")
    print(f"  Context: ...{context}...")
    print()

Postings with zero skills extracted:
    job_id                                                                                                               title       domain
3887883818                                                                  Unix Systems Administrator with Security Clearance           IT
3905309391 Principal Research Scientist – Health Care Quality Measurement: Medicare, Medicaid, & Commercial Insurance Programs Data Science
3905278940                                                                                    Cognitive Performance Specialist           HR
3905283334                                                                                    Cognitive Performance Specialist           HR
3905283332                                                                                    Cognitive Performance Specialist           HR
3905277964                                                                                    Cognitive Performance Special

In [17]:
import re

# rebuilding extraction function with word boundary matching
# short tokens are especially vulnerable to substring false positives
def extract_skills_from_description_v2(description):
    text = str(description).lower()
    matched = []
    for token in canonical_tokens:
        # using word boundary regex for all tokens
        pattern = r'\b' + re.escape(token) + r'\b'
        if re.search(pattern, text):
            matched.append(token)
    return matched

# reapplying extraction
corpus["extracted_skills_raw"] = corpus["description"].apply(
    extract_skills_from_description_v2
)
corpus["normalized_skills"] = corpus["extracted_skills_raw"].apply(normalize_skill_list)
corpus["skill_count"] = corpus["normalized_skills"].apply(len)

corpus["extracted_skills_raw"] = corpus["extracted_skills_raw"].apply(lambda x: "|".join(x))
corpus["normalized_skills"] = corpus["normalized_skills"].apply(lambda x: "|".join(x))

# comparing before and after
print("Skill extraction results after word boundary fix:")
print(f"  Postings with at least one skill: "
      f"{(corpus['skill_count'] > 0).sum()} / {len(corpus)}")
print(f"  Skill count - min/mean/max: "
      f"{corpus['skill_count'].min()} / "
      f"{round(corpus['skill_count'].mean(), 2)} / "
      f"{corpus['skill_count'].max()}")
print()

# checking CAD specifically
cad_after = corpus[corpus["extracted_skills_raw"].str.contains("cad", na=False)]
print(f"CAD extracted after fix: {len(cad_after)} postings")
print(cad_after["domain"].value_counts().to_string())
print()

# updated skill frequency
all_extracted_v2 = []
for skills in corpus["extracted_skills_raw"]:
    if skills:
        all_extracted_v2.extend(skills.split("|"))

skill_freq_v2 = Counter(all_extracted_v2)
print("Top 20 skills after fix:")
for skill, count in skill_freq_v2.most_common(20):
    print(f"  {skill:<35} {count}")
print()

print("Mean skills per domain after fix:")
print(corpus.groupby("domain")["skill_count"].mean().round(2).to_string())

Skill extraction results after word boundary fix:
  Postings with at least one skill: 234 / 252
  Skill count - min/mean/max: 0 / 2.81 / 15

CAD extracted after fix: 10 postings
domain
Engineering    8
IT             1
Management     1

Top 20 skills after fix:
  compliance                          124
  strategy                            105
  recruitment                         50
  agile                               43
  sql                                 39
  python                              37
  machine learning                    35
  risk management                     33
  manufacturing                       33
  aws                                 28
  azure                               20
  java                                15
  linux                               13
  payroll                             13
  employee engagement                 13
  kubernetes                          12
  docker                              10
  cad                                 1

In [18]:
# verifying strategy token — checking if it appears in non-management domains legitimately
strategy_postings = corpus[corpus["extracted_skills_raw"].str.contains("strategy", na=False)]
print(f"Postings with strategy extracted: {len(strategy_postings)}")
print()
print("Strategy by domain:")
print(strategy_postings["domain"].value_counts().to_string())
print()

# spot checking a non-management example
non_mgmt_strategy = strategy_postings[strategy_postings["domain"] != "Management"].head(2)
for _, row in non_mgmt_strategy.iterrows():
    desc_lower = str(row["description"]).lower()
    idx = desc_lower.find("strategy")
    context = desc_lower[max(0, idx-50):idx+60]
    print(f"Title: {row['title']} | Domain: {row['domain']}")
    print(f"  Context: ...{context}...")
    print()

# final corpus column preparation before export
# converting skill_tags and industry_tags lists to pipe-delimited strings
corpus["skill_tags"] = corpus["skill_tags"].apply(
    lambda x: "|".join(x) if isinstance(x, list) else ""
)
corpus["industry_tags"] = corpus["industry_tags"].apply(
    lambda x: "|".join(x) if isinstance(x, list) else ""
)

print("Final corpus shape:", corpus.shape)
print()
print("Final columns:")
for col in corpus.columns:
    null_count = corpus[col].isnull().sum()
    print(f"  {col:<35} nulls: {null_count}")

Postings with strategy extracted: 105

Strategy by domain:
domain
Management      21
IT              18
HR              18
Data Science    17
Legal           16
Engineering     15

Title: Enterprise Platform Solution Architect | Domain: IT
  Context: ...d environment.responsibilities:essential functionsstrategy & planning• develop high-level and detailed solutio...

Title: Oracle Database Administrator | Domain: IT
  Context: ... resolve identified issues.  • design appropriate strategy to reorganize database objects to release unused sp...

Final corpus shape: (252, 13)

Final columns:
  job_id                              nulls: 0
  title                               nulls: 0
  company_name                        nulls: 0
  location                            nulls: 0
  description                         nulls: 0
  formatted_experience_level          nulls: 47
  formatted_work_type                 nulls: 0
  domain                              nulls: 0
  skill_tags                 

In [19]:
import os

output_dir = "../outputs"
os.makedirs(output_dir, exist_ok=True)

# exporting primary corpus artifact
corpus.to_csv(f"{output_dir}/curated_job_descriptions.csv", index=False)
print(f"curated_job_descriptions.csv exported: {corpus.shape}")
print()

# exporting job title taxonomy — unique titles per domain for reference
title_taxonomy = (
    corpus[["domain", "title", "formatted_experience_level"]]
    .sort_values(["domain", "title"])
    .reset_index(drop=True)
)
title_taxonomy.to_csv(f"{output_dir}/job_title_taxonomy.csv", index=False)
print(f"job_title_taxonomy.csv exported: {title_taxonomy.shape}")
print()

# exporting domain distribution summary
domain_summary = corpus.groupby("domain").agg(
    posting_count=("job_id", "count"),
    mean_description_length=("description", lambda x: round(x.str.len().mean())),
    mean_skill_count=("skill_count", lambda x: round(x.mean(), 2)),
    min_skill_count=("skill_count", "min"),
    max_skill_count=("skill_count", "max"),
    pct_with_skills=("skill_count", lambda x: round((x > 0).sum() / len(x) * 100, 1))
).reset_index()

domain_summary.to_csv(f"{output_dir}/domain_distribution_summary.csv", index=False)
print(f"domain_distribution_summary.csv exported: {domain_summary.shape}")
print()
print("Domain distribution summary:")
print(domain_summary.to_string(index=False))
print()

# final validation — reloading all three artifacts
print("Validation — reloading exported artifacts:")
c1 = pd.read_csv(f"{output_dir}/curated_job_descriptions.csv")
c2 = pd.read_csv(f"{output_dir}/job_title_taxonomy.csv")
c3 = pd.read_csv(f"{output_dir}/domain_distribution_summary.csv")
print(f"  curated_job_descriptions.csv:    {c1.shape}")
print(f"  job_title_taxonomy.csv:          {c2.shape}")
print(f"  domain_distribution_summary.csv: {c3.shape}")
print()

# listing all output artifacts
print("All outputs directory contents:")
for f in sorted(os.listdir(output_dir)):
    size_kb = round(os.path.getsize(f"{output_dir}/{f}") / 1024, 1)
    print(f"  {f:<50} {size_kb} KB")

curated_job_descriptions.csv exported: (252, 13)

job_title_taxonomy.csv exported: (252, 3)

domain_distribution_summary.csv exported: (6, 7)

Domain distribution summary:
      domain  posting_count  mean_description_length  mean_skill_count  min_skill_count  max_skill_count  pct_with_skills
Data Science             42                     8576              5.05                0               15             97.6
 Engineering             42                     8467              2.43                0                6             95.2
          HR             42                     9789              2.19                0                5             88.1
          IT             42                     9995              3.12                0                8             92.9
       Legal             42                     8987              1.83                0                4             92.9
  Management             42                    10538              2.24                0         

## Notebook 04 Summary

### Objective Completed
Built a clean, representative job description corpus of 252 postings across six target
domains from the LinkedIn Job Postings 2023-2024 dataset. All three artifacts exported
and validated for downstream use.

### Dataset
LinkedIn Job Postings 2023-2024 (Arsh Kon, Kaggle)
Raw postings: 123,849
After baseline quality filters: 109,874
Final corpus: 252 postings (42 per domain)

### Domain Assignment Approach
Title keyword matching was the primary signal throughout.
Three refinement iterations were required to eliminate noise:
- Iteration 1: baseline keyword lists produced 21,977 Management postings
- Iteration 2: industry exclusion reduced Management but skill tag fallbacks
  introduced production and operations noise
- Iteration 3: IT and Management skill tag fallbacks removed entirely;
  Legal keywords narrowed to prevent counselor and counseling substring matches
HR and Legal skill tag fallbacks were retained as they are domain-specific enough
to be safe.

### Critical Technical Finding
Plain substring matching is not acceptable for skill token extraction.
Word boundary regex matching is mandatory for all token extraction steps.
Substring matching caused aws to match inside draws and laws, and cad to
match inside academic and cascade, inflating extraction counts by 30 to 40 percent.
This finding applies to all downstream notebooks that extract skill tokens from text.

### Skill Extraction
- Vocabulary-driven extraction using 35 canonical tokens from Notebook 03
- Word boundary regex applied to all tokens
- 234 of 252 postings have at least one skill extracted (92.9%)
- 18 zero-skill postings retained — legitimate roles using terminology
  outside the canonical vocabulary; they contribute description text for
  semantic matching

### Corpus Quality
- Mean description length: 9,392 characters
- Minimum description length: 7,105 characters
- Sampling by description length ensured substantive content throughout
- Zero nulls in all critical fields

### Artifacts Produced
- outputs/curated_job_descriptions.csv (252 rows, 13 columns)
- outputs/job_title_taxonomy.csv (252 rows, 3 columns)
- outputs/domain_distribution_summary.csv (6 rows, 7 columns)

### Decisions Deferred
- Vocabulary expansion for novel job description tokens — deferred to Notebook 05
- Experience level null handling — deferred to ATS scoring design
- Embedding precomputation — deferred to dedicated notebook after ATS design

